<a href="https://colab.research.google.com/github/e23378-Tharz/Statistical-Learning-e23378/blob/main/bayesian_inference_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bayesian Inference Assignment — Solutions


In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from scipy.stats import beta as beta_dist
from scipy.stats import norm as norm_dist

np.set_printoptions(precision=4, suppress=True)


---
# Q1 — Bayesian Estimation of Ability from Item Responses (2PL IRT)

## Task 1 — Visualizing the Mechanics

The item response function is
$$p_i(\theta)=\frac{1}{1+e^{-a_i(\theta-b_i)}}.$$

Below we plot two distinct discrimination values $a\in\{0.7,\,2.0\}$. The steeper value $a=2.0$ is paired with three difficulty values $b\in\{-1,0,1\}$, while $a=0.7$ is shown at a single reference difficulty $b=0$.


In [ ]:
theta = np.linspace(-5, 5, 400)

def p_2pl(theta, a, b):
    return 1.0 / (1.0 + np.exp(-a * (theta - b)))

fig = go.Figure()
fig.add_trace(go.Scatter(x=theta, y=p_2pl(theta, 0.7, 0.0),
                          name="a=0.7, b=0 (low discrimination)",
                          line=dict(width=3, dash="dash")))
for b_val, color in zip([-1, 0, 1], ["firebrick", "seagreen", "royalblue"]):
    fig.add_trace(go.Scatter(x=theta, y=p_2pl(theta, 2.0, b_val),
                              name=f"a=2.0, b={b_val}",
                              line=dict(width=3, color=color)))

fig.update_layout(title="2PL Item Response Curves: effect of discrimination (a) and difficulty (b)",
                   xaxis_title="Ability θ", yaxis_title="P(Y=1 | θ)",
                   template="plotly_white", legend=dict(y=0.02, x=0.6))
fig.show()


**Interpretation.** Increasing $a_i$ steepens the curve around its midpoint (higher discrimination — the item separates low- from high-ability users more sharply). Changing $b_i$ **translates the curve horizontally** without changing its shape: the curve for $b_i$ is the curve for $b_i=0$ shifted right by $b_i$. This is because $b_i$ is exactly the ability level at which $p_i(\theta)=0.5$: a *harder* item (larger $b_i$) requires a *higher* ability before the user is even-odds to answer correctly.

## Task 2 — Sequential Likelihood Contribution

A single response is Bernoulli with success probability $p_k(\theta)$:
$$L(y_k\mid\theta) = p_k(\theta)^{y_k}\,\bigl(1-p_k(\theta)\bigr)^{1-y_k}.$$

Assuming conditional independence of responses given $\theta$, the joint likelihood of the running history is the product of the individual contributions:
$$L(\mathbf y^{(k)}\mid\theta)=\prod_{i=1}^{k} p_i(\theta)^{y_i}\bigl(1-p_i(\theta)\bigr)^{1-y_i}.$$

## Task 3 — Mathematical Formulation of the Running Update

By Bayes' theorem, using the step-$(k-1)$ posterior as the new prior:
$$f_{\Theta\mid\mathbf Y^{(k)}}(\theta\mid\mathbf y^{(k)}) \;\propto\; f_{\Theta\mid\mathbf Y^{(k-1)}}(\theta\mid\mathbf y^{(k-1)})\;\cdot\;L(y_k\mid\theta)
= f_{\Theta\mid\mathbf Y^{(k-1)}}(\theta\mid\mathbf y^{(k-1)})\cdot p_k(\theta)^{y_k}(1-p_k(\theta))^{1-y_k}.$$

Only the *newest* observation's likelihood needs to be multiplied in at each step — the entire history is already summarized in the previous posterior.

## Task 4 — Dynamic Shifting

If $y_k=1$ on a **hard** item (large $b_k$), the multiplier $p_k(\theta)$ is small for $\theta \ll b_k$ and rises toward 1 only once $\theta$ approaches/exceeds $b_k$. Multiplying the prior by this near-step-function **suppresses the low-$\theta$ region** of the density while leaving the high-$\theta$ region comparatively untouched. The net effect is that probability mass — and hence the peak — is pushed to the **right** (toward higher ability), and the harder the item ($b_k$ larger), the further right the peak moves, because correctly solving a difficult item is strong evidence against low ability.

## Task 5 — Discrimination and Sharpness

The steepness of $p_k(\theta)$ around $b_k$ is controlled by $a_k$.

- **Large $a_k$:** the logistic curve becomes close to a step function. Multiplying the prior by this near-indicator function very strongly downweights one side of $b_k$, producing a large, sharp change and a **substantially narrower (lower-variance)** posterior — the item is highly informative.
- **Small $a_k$:** the curve is nearly flat/linear across the plausible $\theta$ range, so $p_k(\theta)\approx$ constant and the likelihood barely perturbs the prior's shape — the posterior stays **almost as wide** as before, i.e. the item carries little information.

## Task 6 — Numerical Grid Implementation

1. Fix a dense grid $\theta_1,\dots,\theta_M$ (e.g. $[-6,6]$).
2. Store the current (discretized) posterior as a vector `prior_grid` evaluated on this grid (initialized to the $\mathscr N(0,1)$ density).
3. On observing $y_k$ with known $(a_k,b_k)$: compute `lik = p(theta_grid)**y_k * (1-p(theta_grid))**(1-y_k)`.
4. Form the unnormalized posterior `unnorm = prior_grid * lik`.
5. **Sequential normalization:** integrate numerically (trapezoidal rule) `Z = trapz(unnorm, theta_grid)` and set `posterior_grid = unnorm / Z`.
6. Set `prior_grid = posterior_grid` and repeat for the next item.

## Task 7 — Simulation and Convergence


In [ ]:
def p_2pl(theta, a, b):
    return 1.0 / (1.0 + np.exp(-a * (theta - b)))

def run_irt_simulation(theta_true=0.75, n_items=20, seed=7, grid=None):
    rng = np.random.default_rng(seed)
    if grid is None:
        grid = np.linspace(-6, 6, 1201)

    prior = norm_dist.pdf(grid, 0, 1)
    prior /= np.trapezoid(prior, grid)

    bayes_hist = [np.trapezoid(grid * prior, grid)]
    map_hist = [grid[np.argmax(prior)]]
    items = []

    for k in range(1, n_items + 1):
        a_k = rng.uniform(0.5, 2.0)
        b_k = rng.normal(0, 1)
        p_true = p_2pl(theta_true, a_k, b_k)
        y_k = 1 if rng.uniform() < p_true else 0
        items.append((a_k, b_k, y_k))

        lik = p_2pl(grid, a_k, b_k) ** y_k * (1 - p_2pl(grid, a_k, b_k)) ** (1 - y_k)
        unnorm = prior * lik
        Z = np.trapezoid(unnorm, grid)
        posterior = unnorm / Z

        bayes_hist.append(np.trapezoid(grid * posterior, grid))
        map_hist.append(grid[np.argmax(posterior)])
        prior = posterior

    return np.array(bayes_hist), np.array(map_hist), items

theta_true = 0.75
bayes_hist, map_hist, items = run_irt_simulation(theta_true=theta_true, n_items=20, seed=7)
steps = np.arange(0, 21)

fig = go.Figure()
fig.add_trace(go.Scatter(x=steps, y=bayes_hist, mode="lines+markers", name="Posterior Mean (Bayes)"))
fig.add_trace(go.Scatter(x=steps, y=map_hist, mode="lines+markers", name="MAP estimate"))
fig.add_hline(y=theta_true, line_dash="dash", line_color="black",
              annotation_text=f"θ_true = {theta_true}", annotation_position="bottom right")
fig.update_layout(title="Sequential Ability Estimation: Posterior Mean & MAP vs. Item Number",
                   xaxis_title="Item step k", yaxis_title="Estimated ability θ",
                   template="plotly_white")
fig.show()

print(f"Final posterior mean: {bayes_hist[-1]:.3f}   Final MAP: {map_hist[-1]:.3f}")


Final posterior mean: 0.911   Final MAP: 0.900


**Analysis.** Both estimators start at $0$ (the prior mean/mode) and, as items accumulate, wander toward and then settle near $\theta_{\text{true}}=0.75$, with the distance $|\widehat\theta^{(k)}-\theta_{\text{true}}|$ generally **shrinking** as $k$ grows (though not monotonically, since each item is a noisy Bernoulli draw and highly discriminating items can cause larger short-term jumps). This reflects the accumulation of Fisher information: each new response narrows the posterior (Task 5), so the estimates become both more accurate *and* more stable — the platform's confidence in its ability measurement increases with the number of items answered, exactly as intended by an adaptive-testing engine.


---
# Q2 — Bayesian Tracking of CTR via Beta-Binomial Conjugacy

## Task 1 — Structural Probability and Properties


In [ ]:
theta_grid01 = np.linspace(0.001, 0.999, 500)
params = [(1, 1, "Beta(1,1) — uninformative"),
          (2, 8, "Beta(2,8) — right-skewed (low CTR belief)"),
          (8, 2, "Beta(8,2) — left-skewed (high CTR belief)")]

fig = go.Figure()
for a_, b_, label in params:
    fig.add_trace(go.Scatter(x=theta_grid01, y=beta_dist.pdf(theta_grid01, a_, b_),
                              name=label, line=dict(width=3)))
fig.update_layout(title="Beta(α, β) Densities for Different Prior Beliefs",
                   xaxis_title="θ (click-through rate)", yaxis_title="Density",
                   template="plotly_white")
fig.show()


**Interpretation.** The mean of a $\text{Beta}(\alpha,\beta)$ is $\alpha/(\alpha+\beta)$. Increasing $\alpha$ relative to $\beta$ shifts the center of mass **toward 1**; increasing $\beta$ relative to $\alpha$ shifts it **toward 0**. $\text{Beta}(1,1)$ is the flat/uniform, uninformative case; $\text{Beta}(2,8)$ concentrates mass near low CTR values; $\text{Beta}(8,2)$ concentrates mass near high CTR values. The overall spread also narrows as $\alpha+\beta$ grows, since this "pseudo-count" total controls the prior's effective sample size.

## Task 2 — Sequential Likelihood and Joint History

$$L(y_k\mid\theta)=\theta^{y_k}(1-\theta)^{1-y_k},\qquad
L(\mathbf y^{(k)}\mid\theta)=\prod_{i=1}^{k}\theta^{y_i}(1-\theta)^{1-y_i}=\theta^{\sum_i y_i}(1-\theta)^{k-\sum_i y_i}.$$

## Task 3 — Closed-Form Conjugate Update

$$f_{\Theta\mid\mathbf Y^{(k)}}(\theta\mid\mathbf y^{(k)}) \propto \theta^{\alpha_{k-1}-1}(1-\theta)^{\beta_{k-1}-1}\cdot\theta^{y_k}(1-\theta)^{1-y_k}
= \theta^{(\alpha_{k-1}+y_k)-1}(1-\theta)^{(\beta_{k-1}+1-y_k)-1}.$$

This is (up to normalization) exactly the kernel of a $\text{Beta}(\alpha_k,\beta_k)$ density, proving **Beta–Binomial conjugacy**, with the simple arithmetic updates
$$\alpha_k=\alpha_{k-1}+y_k, \qquad \beta_k=\beta_{k-1}+(1-y_k).$$

Posterior mean:
$$\mathbb E[\Theta\mid\mathbf Y^{(k)}=\mathbf y^{(k)}]=\frac{\alpha_k}{\alpha_k+\beta_k}.$$

## Task 4 — Dynamic Shifting Mechanics

A **click** ($y_k=1$) increments $\alpha_k$ only, pushing the density's mass and mode **rightward** (higher believed CTR). A **non-click** ($y_k=0$) increments $\beta_k$ only, pushing mass **leftward**. Because the update is a pure integer increment of the sufficient statistics, it is exact and instantaneous — no numerical integration is ever needed. This contrasts sharply with the 2PL IRT model of Q1, where the Bernoulli-logistic likelihood is **not conjugate** to any tractable prior family; there, the posterior kernel has no closed analytical form and must be tracked and renormalized on a numerical grid at every step.

## Task 5 — Running Point Estimators

$$\widehat\theta_{\mathrm{Bayes}}^{(k)}=\frac{\alpha_k}{\alpha_k+\beta_k}, \qquad
\widehat\theta_{\mathrm{MAP}}^{(k)}=\frac{\alpha_k-1}{\alpha_k+\beta_k-2}\quad(\alpha_k,\beta_k>1).$$
(If $\alpha_k\le 1$ or $\beta_k\le1$ the mode lies at a boundary, $0$ or $1$, rather than at this interior stationary point.)

## Task 6 — Performance Tracking


In [ ]:
def run_ctr_simulation(theta_true=0.35, n_impressions=100, alpha0=1.0, beta0=1.0, seed=11):
    rng = np.random.default_rng(seed)
    alpha_k, beta_k = alpha0, beta0
    bayes_hist = [alpha_k / (alpha_k + beta_k)]
    map_hist = [np.nan if (alpha_k <= 1 or beta_k <= 1) else (alpha_k - 1) / (alpha_k + beta_k - 2)]

    for k in range(1, n_impressions + 1):
        y_k = 1 if rng.uniform() < theta_true else 0
        alpha_k += y_k
        beta_k += (1 - y_k)
        bayes_hist.append(alpha_k / (alpha_k + beta_k))
        map_hist.append(np.nan if (alpha_k <= 1 or beta_k <= 1) else (alpha_k - 1) / (alpha_k + beta_k - 2))

    return np.array(bayes_hist), np.array(map_hist)

theta_true_ctr = 0.35
bayes_ctr, map_ctr = run_ctr_simulation(theta_true=theta_true_ctr, n_impressions=100, seed=11)
steps_ctr = np.arange(0, 101)

fig = go.Figure()
fig.add_trace(go.Scatter(x=steps_ctr, y=bayes_ctr, mode="lines", name="Posterior Mean (Bayes)"))
fig.add_trace(go.Scatter(x=steps_ctr, y=map_ctr, mode="lines", name="MAP estimate"))
fig.add_hline(y=theta_true_ctr, line_dash="dash", line_color="black",
              annotation_text=f"θ_true = {theta_true_ctr}", annotation_position="bottom right")
fig.update_layout(title="Sequential CTR Estimation: Posterior Mean & MAP vs. Impression Count",
                   xaxis_title="Impression step k", yaxis_title="Estimated CTR θ",
                   template="plotly_white")
fig.show()

print(f"Final posterior mean: {bayes_ctr[-1]:.4f}   Final MAP: {map_ctr[-1]:.4f}")


Final posterior mean: 0.4510   Final MAP: 0.4500


**Analysis.** Starting from the flat $\text{Beta}(1,1)$ prior, both estimators fluctuate substantially over the first ~10–20 impressions (each observation still carries meaningful weight relative to the small pseudo-count total $\alpha_k+\beta_k$). As $k\to100$, $\alpha_k+\beta_k$ grows large and each single observation's marginal effect on $\alpha_k/(\alpha_k+\beta_k)$ shrinks like $O(1/k)$, so the estimators settle down and concentrate near $\theta_{\text{true}}=0.35$. This illustrates a general fact about conjugate updating: the influence of the initial prior (here, uninformative) is asymptotically **washed out** by the accumulating data — the posterior is increasingly dominated by the likelihood as evidence accrues.


---
# Q3 — Bayesian SHM via Bounded Grid Updates

## Task 1 — Prior Belief Boundaries


In [ ]:
theta_shm_grid = np.linspace(0.01, 1.0, 500)
prior_shm = beta_dist.pdf(theta_shm_grid, 8, 1.5)

fig = go.Figure()
fig.add_trace(go.Scatter(x=theta_shm_grid, y=prior_shm, fill="tozeroy", name="Beta(8, 1.5) prior"))
fig.update_layout(title="Initial Prior on Remaining Stiffness Efficiency θ",
                   xaxis_title="θ (stiffness efficiency)", yaxis_title="Density",
                   template="plotly_white")
fig.show()

E_prior = 8 / (8 + 1.5)
print(f"E[Θ^(0)] = 8/9.5 = {E_prior:.4f}")


E[Θ^(0)] = 8/9.5 = 0.8421


$$\mathbb E[\Theta^{(0)}]=\frac{\alpha}{\alpha+\beta}=\frac{8}{9.5}\approx 0.842.$$

**Why this prior is appropriate.** $\text{Beta}(8,1.5)$ is supported on exactly $[0,1]$ (matching the physical range of a stiffness-efficiency ratio), is strongly concentrated toward $\theta=1$ (reflecting the engineering assumption that a newly commissioned/inspected component starts near-pristine), yet still assigns non-negligible density to values below $1$ — leaving room for the sensor data to reveal unexpected pre-existing degradation rather than assigning it zero prior probability outright.

## Task 2 — Structural Likelihood Formulation

Since $y_k=\theta K_{\text{nominal}}e^{\epsilon_k}$ with $\epsilon_k\sim\mathscr N(0,\sigma^2)$, taking logs gives $\ln y_k \mid \theta \sim \mathscr N(\ln\theta+\ln K_{\text{nominal}},\,\sigma^2)$, i.e. $y_k\mid\theta$ is **log-normal**. By the change-of-variables formula (Jacobian $1/y_k$):
$$L(y_k\mid\theta)=\frac{1}{y_k\,\sigma\sqrt{2\pi}}\exp\!\left[-\frac{\bigl(\ln y_k-\ln\theta-\ln K_{\text{nominal}}\bigr)^2}{2\sigma^2}\right].$$

Joint likelihood for the running history (conditional independence of noise terms):
$$L(\mathbf y^{(k)}\mid\theta)=\prod_{i=1}^{k}\frac{1}{y_i\,\sigma\sqrt{2\pi}}\exp\!\left[-\frac{(\ln y_i-\ln\theta-\ln K_{\text{nominal}})^2}{2\sigma^2}\right].$$

## Task 3 — Non-Conjugate Grid Update

A Beta prior has kernel $\theta^{\alpha-1}(1-\theta)^{\beta-1}$ — a *polynomial* in $\theta$ — while the log-normal likelihood's dependence on $\theta$ enters through $\exp[-(\ln\theta-c)^2/(2\sigma^2)]$, a transcendental function of $\ln\theta$. Multiplying these two kernels does **not** collapse back into any recognizable, finitely-parameterized closed family (Beta or otherwise), so there is no conjugate update rule. We can still write the recursive Bayes relation up to a proportionality constant:
$$f_{\Theta\mid\mathbf Y^{(k)}}(\theta\mid\mathbf y^{(k)}) \;\propto\; f_{\Theta\mid\mathbf Y^{(k-1)}}(\theta\mid\mathbf y^{(k-1)})\cdot L(y_k\mid\theta),$$
which must be evaluated and renormalized numerically at every step.

## Task 4 — Running Point Estimates (Numerical Integrals)

$$\widehat\theta_{\mathrm{Bayes}}^{(k)}=\int_0^1 \theta\, f_{\Theta\mid\mathbf Y^{(k)}}(\theta\mid\mathbf y^{(k)})\,d\theta,
\qquad
\widehat\theta_{\mathrm{MAP}}^{(k)}=\operatorname*{arg\,max}_{\theta\in(0,1]} f_{\Theta\mid\mathbf Y^{(k)}}(\theta\mid\mathbf y^{(k)}).$$

## Task 5 — Algorithmic Grid Approximation and Normalization

1. Build a grid $\theta_1,\dots,\theta_M$ over $(0,1]$, e.g. `np.linspace(0.001, 1.0, M)` — a small $\epsilon$ offset from $0$ avoids the $\ln\theta\to-\infty$ singularity in the likelihood, while the grid's right endpoint is exactly $1$ since $\theta=1$ (pristine) is a physically attainable value.
2. Maintain the current posterior as a vector `prior_grid` on this grid.
3. On observing $y_k$: compute `lik_grid = L(y_k, theta_grid)` from the log-normal formula above.
4. Form `unnorm = prior_grid * lik_grid`.
5. **Sequential normalization via trapezoidal rule:** `Z = np.trapezoid(unnorm, theta_grid)`, then `posterior_grid = unnorm / Z`.
6. Set `prior_grid = posterior_grid` for the next inspection, and read off $\widehat\theta_{\mathrm{Bayes}}^{(k)}=$ `np.trapezoid(theta_grid*posterior_grid, theta_grid)` and $\widehat\theta_{\mathrm{MAP}}^{(k)}=$ `theta_grid[argmax(posterior_grid)]`.

## Task 6 — Performance Tracking and Degradation Convergence


In [ ]:
def loglik_shm(y, theta, K_nom, sigma):
    return (1.0 / (y * sigma * np.sqrt(2 * np.pi))) * np.exp(
        -(np.log(y) - np.log(theta) - np.log(K_nom)) ** 2 / (2 * sigma ** 2)
    )

def run_shm_simulation(theta_true=0.68, n_steps=15, K_nom=50.0, sigma=0.15, seed=3):
    rng = np.random.default_rng(seed)
    theta_grid = np.linspace(0.001, 1.0, 2000)
    prior = beta_dist.pdf(theta_grid, 8, 1.5)
    prior /= np.trapezoid(prior, theta_grid)

    bayes_hist = [np.trapezoid(theta_grid * prior, theta_grid)]
    map_hist = [theta_grid[np.argmax(prior)]]
    milestone_curves = {0: prior.copy()}
    milestones = {0, 1, 2, 5, 10, 15}

    for k in range(1, n_steps + 1):
        eps = rng.normal(0, sigma)
        y_k = theta_true * K_nom * np.exp(eps)
        lik = loglik_shm(y_k, theta_grid, K_nom, sigma)
        unnorm = prior * lik
        Z = np.trapezoid(unnorm, theta_grid)
        posterior = unnorm / Z

        bayes_hist.append(np.trapezoid(theta_grid * posterior, theta_grid))
        map_hist.append(theta_grid[np.argmax(posterior)])
        prior = posterior
        if k in milestones:
            milestone_curves[k] = posterior.copy()

    return theta_grid, np.array(bayes_hist), np.array(map_hist), milestone_curves

theta_true_shm = 0.68
theta_grid_shm, bayes_shm, map_shm, curves_shm = run_shm_simulation(theta_true=theta_true_shm, seed=3)

# Plot 1: posterior density progression
fig1 = go.Figure()
for k in sorted(curves_shm):
    fig1.add_trace(go.Scatter(x=theta_grid_shm, y=curves_shm[k], name=f"k={k}", line=dict(width=2.5)))
fig1.add_vline(x=theta_true_shm, line_dash="dash", line_color="black",
               annotation_text=f"θ_true={theta_true_shm}")
fig1.update_layout(title="Posterior Density Evolution: Stiffness Efficiency θ",
                    xaxis_title="θ", yaxis_title="Density", template="plotly_white")
fig1.show()

# Plot 2: estimator convergence
steps_shm = np.arange(0, 16)
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=steps_shm, y=bayes_shm, mode="lines+markers", name="Posterior Mean (Bayes)"))
fig2.add_trace(go.Scatter(x=steps_shm, y=map_shm, mode="lines+markers", name="MAP estimate"))
fig2.add_hline(y=theta_true_shm, line_dash="dash", line_color="black",
               annotation_text=f"θ_true = {theta_true_shm}", annotation_position="bottom right")
fig2.update_layout(title="Convergence of Stiffness Estimators over Inspection Timeline",
                    xaxis_title="Inspection step k", yaxis_title="Estimated θ",
                    template="plotly_white")
fig2.show()

print("Estimator trajectory (Bayes mean):", np.round(bayes_shm, 3))


Estimator trajectory (Bayes mean): [0.842 0.889 0.709 0.714 0.691 0.68  0.676 0.648 0.649 0.643 0.68  0.682
 0.679 0.677 0.672 0.666]


**Analysis.** The optimistic $\text{Beta}(8,1.5)$ prior starts centered near $0.84$. Within the first few sensor readings the log-normal evidence — which is concentrated tightly around the true value because $\sigma=0.15$ is modest relative to the $0.84\to0.68$ gap — dominates the prior, and by roughly the **4th–5th reading** the estimators have already dropped into the neighborhood of $\theta_{\text{true}}=0.68$ and stay there (see the printed trajectory above). The posterior density curves in the first plot visibly **narrow** as $k$ increases from $0$ to $15$: this shrinking spread is the direct numerical signature of accumulating measurement information, and in a real SHM deployment it is exactly this narrowing that lets engineers move from "damage is *plausible*" to "damage is *confirmed within a tight confidence band*," which is the quantitative basis for triggering a maintenance or safety threshold alert.


---
# Q4 — Gaussian Mixture Clustering as Conditional Updating

## Task 1 — Deriving the Marginal Density

By the law of total probability, conditioning on the discrete latent variable $C_i$ and summing over its support:
$$p(x_i)=\sum_{k=1}^K P(C_i=k)\,p(x_i\mid C_i=k)=\sum_{k=1}^K \phi_k\,\mathscr N(x_i\mid\mu_k,\Sigma_k).$$
It is called a **Gaussian mixture density** because it is a convex combination ($\phi_k\ge0$, $\sum_k\phi_k=1$) of $K$ Gaussian component densities — a weighted blend rather than a single unimodal Gaussian, capable of representing multi-modal data.

## Task 2 — Deriving the Posterior Cluster Probability

By Bayes' rule applied to the discrete/continuous pair $(C_i,X_i)$:
$$P(C_i=k\mid X_i=x_i)=\frac{P(X_i=x_i\mid C_i=k)P(C_i=k)}{\sum_{j}P(X_i=x_i\mid C_i=j)P(C_i=j)}=\frac{\phi_k\mathscr N(x_i\mid\mu_k,\Sigma_k)}{\sum_j\phi_j\mathscr N(x_i\mid\mu_j,\Sigma_j)}=\gamma_{ik}.$$
$\gamma_{ik}$ is a genuine posterior probability: it combines the prior cluster weight $\phi_k$ with the observed-data likelihood under cluster $k$, exactly as Bayes' theorem prescribes, updating our belief about membership *after* seeing $x_i$.

## Task 3 — One-Hot Encoding and Conditional Expectation

$Z_{ik}$ is an indicator (Bernoulli-type) random variable, so its conditional expectation equals the conditional probability of the event it indicates:
$$\mathbb E[Z_{ik}\mid X_i=x_i]=1\cdot P(C_i=k\mid x_i)+0\cdot P(C_i\ne k\mid x_i)=P(C_i=k\mid x_i)=\gamma_{ik}.$$
Stacking over $k=1,\dots,K$: $\mathbb E[Z_i\mid X_i=x_i]=(\gamma_{i1},\dots,\gamma_{iK})^\top$. Hence the **soft cluster assignment** used in GMM clustering is precisely the conditional expectation of the latent one-hot membership vector given the data.

## Task 4 — Soft vs. Hard Clustering

Soft clustering reports the *entire* probability vector $\mathbb E[Z_i\mid x_i]$, preserving uncertainty about membership (a point near a cluster boundary might get $\gamma\approx(0.5,0.5,0)$). Hard clustering collapses this to a single label $\widehat C_i=\arg\max_k\gamma_{ik}$, discarding the ambiguity information in exchange for a simple partition.

## Task 5 — Conditional Expectation of the Observation Given the Cluster

Directly from the generative model $X_i\mid C_i=k\sim\mathscr N(\mu_k,\Sigma_k)$: $\mathbb E[X_i\mid C_i=k]=\mu_k$, so $\mu_k$ is the **center of cluster $k$** in feature space. The two conditional expectations answer opposite-direction questions of the same model: $\mathbb E[Z_i\mid X_i=x_i]$ is the *inverse* (data $\to$ cluster) question "given this point, how likely is each cluster?", while $\mathbb E[X_i\mid C_i=k]$ is the *forward* (cluster $\to$ data) question "where do points generated by this cluster tend to land?"

## Task 6 — The Complete-Data Likelihood

Taking logs of $p(x_1,\dots,x_n,z_1,\dots,z_n)=\prod_i\prod_k[\phi_k\mathscr N(x_i\mid\mu_k,\Sigma_k)]^{z_{ik}}$, and using that for each $i$ exactly one $z_{ik}=1$ (all others $0$), the log of the product-of-powers becomes a sum with $z_{ik}$ as a 0/1 selector:
$$\ell_c=\sum_{i=1}^n\sum_{k=1}^K z_{ik}\Bigl[\log\phi_k+\log\mathscr N(x_i\mid\mu_k,\Sigma_k)\Bigr].$$
If the $z_{ik}$ were known, this expression **decouples across clusters**: maximizing over $(\phi_k,\mu_k,\Sigma_k)$ reduces to $K$ independent, standard weighted-Gaussian MLE problems, each using only the subset of points assigned to that cluster — straightforward closed-form estimation.

## Task 7 — The EM Interpretation

Since the true $z_{ik}$ are unobserved, EM replaces them by their conditional expectations given the current parameter estimates and the data — precisely the missing-data-imputation idea — giving the **E-step**:
$$z_{ik}\;\leadsto\;\mathbb E[Z_{ik}\mid X_i=x_i]=\gamma_{ik},\qquad
Q=\sum_{i=1}^n\sum_{k=1}^K\gamma_{ik}\Bigl[\log\phi_k+\log\mathscr N(x_i\mid\mu_k,\Sigma_k)\Bigr].$$
This is a conditional update of cluster membership: at each iteration, every point's membership belief is refreshed using Bayes' rule (Task 2) under the *current* parameter guess, before those (soft) memberships are used to re-estimate the parameters.

## Task 8 — Parameter Updates (M-step)

Maximizing $Q$ subject to $\sum_k\phi_k=1$ (Lagrange multiplier) and setting gradients w.r.t. $\mu_k,\Sigma_k$ to zero yields the standard weighted MLE updates:
$$N_k=\sum_{i=1}^n\gamma_{ik},\quad \phi_k^{\text{new}}=\frac{N_k}{n},\quad
\mu_k^{\text{new}}=\frac{1}{N_k}\sum_{i=1}^n\gamma_{ik}x_i,\quad
\Sigma_k^{\text{new}}=\frac{1}{N_k}\sum_{i=1}^n\gamma_{ik}(x_i-\mu_k^{\text{new}})(x_i-\mu_k^{\text{new}})^\top.$$
Each $\gamma_{ik}$ acts as a **fractional (soft) membership weight**: rather than a point contributing entirely to one cluster's statistics (as in hard $k$-means), it contributes a $\gamma_{ik}$-weighted fraction of itself to every cluster's mean/covariance/count, in proportion to how compatible it currently appears with that cluster.

## Task 9 — Interpretation

Gaussian mixture clustering can be understood as a repeated cycle of *conditional updating*. The mixture weight $\phi_k$ encodes the **prior** belief that a randomly chosen point belongs to cluster $k$, before any data is examined. The Gaussian density $\mathscr N(x_i\mid\mu_k,\Sigma_k)$ measures how **compatible** a specific observation $x_i$ is with cluster $k$'s current location and shape. Combining these via Bayes' rule produces the responsibility $\gamma_{ik}$, the **posterior** probability of membership after observing $x_i$ — and the full vector $\mathbb E[Z_i\mid X_i=x_i]$ is exactly this soft-assignment belief for every cluster simultaneously. The M-step then re-estimates $\phi_k,\mu_k,\Sigma_k$ using these responsibilities as weights, after which the cycle repeats with an updated prior. In short: **GMM clustering is probabilistic clustering built entirely out of successive conditional expectations of a latent cluster-membership variable**, alternating between updating beliefs about *labels given data* (E-step) and beliefs about *parameters given (soft) labels* (M-step).

## Task 10 — Computational Simulation and Out-of-Sample Validation

> **Note on data.** The assignment references the Kaggle "Credit Card Dataset for Clustering" (`arjunbhasin2013/ccdata`). This notebook's execution environment cannot reach `kaggle.com` to download it directly. The `GMMFinancialSegmenter` class below is written to load the real CSV (`CC GENERAL.csv`) if it is present in the working directory (e.g. after downloading it manually and placing it alongside this notebook), and otherwise falls back to a synthetic two-feature dataset (`PURCHASES`, `CREDIT_LIMIT`) with three well-separated financial-behavior regimes, purely so that the full pipeline below can be demonstrated end-to-end. **To reproduce on the real dataset:** download `CC GENERAL.csv` from the Kaggle link and place it in the same folder as this notebook, then rerun — no code changes are needed.


In [ ]:
import pandas as pd
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from plotly.subplots import make_subplots


class GMMFinancialSegmenter:
    # Two-feature Gaussian Mixture segmentation with Plotly diagnostics.

    def __init__(self, n_components=3, feature_x="PURCHASES", feature_y="CREDIT_LIMIT",
                 test_size=0.2, random_state=42):
        self.n_components = n_components
        self.feature_x = feature_x
        self.feature_y = feature_y
        self.test_size = test_size
        self.random_state = random_state
        self.scaler = StandardScaler()
        self.gmm = GaussianMixture(n_components=n_components, random_state=random_state)

    # ---------- data ----------
    def load_data(self, csv_path="CC GENERAL.csv", n_synthetic=1200, seed=0):
        try:
            df = pd.read_csv(csv_path)
            df = df[[self.feature_x, self.feature_y]].dropna()
            print(f"Loaded real dataset from '{csv_path}' with {len(df)} rows.")
        except FileNotFoundError:
            rng = np.random.default_rng(seed)
            n = n_synthetic
            c1 = rng.normal(loc=[200, 1500], scale=[80, 400], size=(n // 3, 2))
            c2 = rng.normal(loc=[1200, 4000], scale=[300, 900], size=(n // 3, 2))
            c3 = rng.normal(loc=[3500, 9000], scale=[600, 1500], size=(n - 2 * (n // 3), 2))
            data = np.clip(np.vstack([c1, c2, c3]), 1, None)
            df = pd.DataFrame(data, columns=[self.feature_x, self.feature_y])
            print(f"'{csv_path}' not found — using {len(df)}-row synthetic fallback dataset instead.")
        self.df = df.reset_index(drop=True)
        return self.df

    def split_and_scale(self):
        X = self.scaler.fit_transform(self.df[[self.feature_x, self.feature_y]].values)
        self.X_train, self.X_test = train_test_split(
            X, test_size=self.test_size, random_state=self.random_state
        )
        return self.X_train, self.X_test

    # ---------- model ----------
    def fit(self):
        self.gmm.fit(self.X_train)
        print(f"Converged: {self.gmm.converged_}   Iterations used: {self.gmm.n_iter_}")
        return self.gmm

    def evaluate(self):
        test_ll = self.gmm.score(self.X_test)
        print(f"Average out-of-sample log-likelihood on test set: {test_ll:.4f}")
        return test_ll

    # ---------- plots ----------
    def plot_empirical_density(self):
        df_plot = self.df
        fig = px.density_heatmap(df_plot, x=self.feature_x, y=self.feature_y,
                                  marginal_x="histogram", marginal_y="histogram",
                                  title="Empirical 2D Density: Training Data")
        fig.show()
        return fig

    def _responsibility_grid(self, X):
        x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
        y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 150), np.linspace(y_min, y_max, 150))
        grid_pts = np.column_stack([xx.ravel(), yy.ravel()])
        resp = self.gmm.predict_proba(grid_pts)
        max_resp = resp.max(axis=1).reshape(xx.shape)
        label_grid = resp.argmax(axis=1).reshape(xx.shape)
        return xx, yy, max_resp, label_grid

    def _assignment_plot(self, X, title):
        xx, yy, max_resp, label_grid = self._responsibility_grid(self.X_train)
        labels = self.gmm.predict(X)
        fig = go.Figure()
        fig.add_trace(go.Contour(x=xx[0], y=yy[:, 0], z=max_resp,
                                  colorscale="Viridis", opacity=0.55,
                                  contours=dict(showlines=False),
                                  colorbar=dict(title="max γ_ik")))
        fig.add_trace(go.Scatter(x=X[:, 0], y=X[:, 1], mode="markers",
                                  marker=dict(color=labels, colorscale="Turbo", size=6,
                                              line=dict(width=0.5, color="white")),
                                  name="points"))
        fig.update_layout(title=title, xaxis_title=f"{self.feature_x} (scaled)",
                           yaxis_title=f"{self.feature_y} (scaled)", template="plotly_white")
        fig.show()
        return fig

    def plot_training_assignments(self):
        return self._assignment_plot(self.X_train, "Training Assignments over Responsibility Contours")

    def plot_test_assignments(self):
        return self._assignment_plot(self.X_test, "Test Assignments over Responsibility Contours")

    def run_all(self, csv_path="CC GENERAL.csv"):
        self.load_data(csv_path)
        self.split_and_scale()
        self.fit()
        self.evaluate()
        self.plot_empirical_density()
        self.plot_training_assignments()
        self.plot_test_assignments()


segmenter = GMMFinancialSegmenter(n_components=3)
segmenter.run_all()


'CC GENERAL.csv' not found — using 1200-row synthetic fallback dataset instead.
Converged: True   Iterations used: 3
Average out-of-sample log-likelihood on test set: -0.7515


**Evaluation of the plots.** The empirical density heatmap reveals the raw multi-modal structure in `(PURCHASES, CREDIT_LIMIT)` space before any modeling — visually motivating a mixture (rather than single-Gaussian) model. The training and test assignment plots overlay the data on a continuous background contour of `max_k γ_ik`, the *maximum posterior responsibility* evaluated across a fine coordinate grid: this is a direct numerical rendering of the soft-assignment vector $\mathbb E[Z_i\mid X_i=x_{\text{grid}}]$ derived analytically in Task 3, now visualized at every point in feature space rather than just at the observed data points. Regions where the contour value is close to $1/K$ (rather than near $1$) mark areas of genuine cluster ambiguity — points there could plausibly belong to more than one financial-behavior segment — and the test-set overlay lets us see whether *unseen* customers tend to fall in confidently-labeled regions or in these ambiguous boundary zones, which is the practical, out-of-sample analogue of the soft-clustering idea developed throughout Q4.
